# 06 — HDR by the Linear Digital Image Composer (LDIC)

Paper Section 3.3.2 (Druckmüllerová 2014): weighted average of the nine
exposures with intensity‑dependent weights (0 below 2 %, ramp to 1 at 15 %,
ramp down 65→80 %, bypassed at the ends of the bracket), per‑exposure linear
gain $k_i(\phi)$ fitted in 60 angular sectors through the origin ($q_i=0$) and
smoothed with a 4th‑order trigonometric polynomial.  The weight function is
evaluated on a **master luminance** (max of the three polariser channels) so all
channels hand over at the same pixels.

Output: `products/hdr/hdr_ldic_pol{1,2,3}.fits` + preview.
Legacy source: `make_ldic_hdr_from_stacked_exposures_Chaitanya.ipynb`.
⏱ several minutes per polariser.

In [ ]:
import sys, time
sys.path.insert(0, "..")          # config.py / utils.py live one level up
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

import config, utils
%matplotlib inline

In [ ]:
from scipy import ndimage

def load_plane(inv_exp, i, pos):
    d = np.asarray(fits.getdata(utils.stacked_filename(inv_exp), memmap=True)[i], dtype=np.float32)
    return ndimage.shift(d, config.CHANNEL_SHIFTS[pos], order=1)

# Pass 1: global scale per channel (prepare_images_for_ldic uses ONE scale per channel across all exposures)
gmax = {}
for i, pos in enumerate(config.POLARIZER_POSITIONS):
    gmax[pos] = max(float(np.clip(load_plane(e, i, pos), 0, None).max()) for e in config.INV_EXPOSURES)
scale = {pos: config.LDIC_TARGET_MAX / gmax[pos] for pos in gmax}
print("global max per channel:", gmax)

In [ ]:
# Pass 2: master luminance reference per exposure = max over the three scaled channels
refs = []
for e in config.INV_EXPOSURES:
    r = None
    for i, pos in enumerate(config.POLARIZER_POSITIONS):
        p = np.clip(np.clip(load_plane(e, i, pos), 0, None) * scale[pos], 0, config.LDIC_TARGET_MAX).astype(np.float32)
        r = p if r is None else np.maximum(r, p)
    refs.append(r)
print(len(refs), "reference images")

In [ ]:
# Exposure-ratio diagnostic (ratio ~0.5 per doubling = linear; ~1 = saturated)
for i, pos in enumerate(config.POLARIZER_POSITIONS):
    mx = [float(load_plane(e, i, pos).max()) for e in config.INV_EXPOSURES]
    print(f"pol{pos}: " + "  ".join(f"1/{config.INV_EXPOSURES[j]}->1/{config.INV_EXPOSURES[j+1]}: {mx[j]/mx[j+1]:.2f}" for j in range(len(mx)-1)))

In [ ]:
sun_cx, sun_cy = config.SUN_CENTER_XY
template = fits.getheader(utils.stacked_filename(config.INV_EXPOSURES[0]))
hdr_planes = {}
for i, pos in enumerate(config.POLARIZER_POSITIONS):
    t0 = time.time()
    imgs = [np.clip(np.clip(load_plane(e, i, pos), 0, None) * scale[pos], 0, config.LDIC_TARGET_MAX).astype(np.float32)
            for e in config.INV_EXPOSURES]
    hdr_planes[pos] = utils.ldic_hdr_stacking(imgs, config.EXPOSURE_TIMES_S, images_ref=refs,
                                              max_pixel_value_input=config.LDIC_TARGET_MAX,
                                              sun_center_x=sun_cx, sun_center_y=sun_cy, verbose=True,
                                              **config.LDIC_PARAMS)
    del imgs
    fits.writeto(utils.hdr_filename("ldic", pos), hdr_planes[pos], header=utils.hdr_header(template, pos, "ldic"), overwrite=True)
    print(f"pol{pos}: min {hdr_planes[pos].min():.4g} max {hdr_planes[pos].max():.4g}  ({time.time()-t0:.0f} s)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, pos in zip(axes, config.POLARIZER_POSITIONS):
    ax.imshow(utils.asinh_stretch(hdr_planes[pos][::4, ::4]), cmap="gray"); ax.set_title(f"LDIC HDR pol{pos} (asinh)"); ax.axis("off")
plt.tight_layout()

In [ ]:
from PIL import Image
rgb = np.stack([utils.normalise_channel(hdr_planes[p]) for p in config.POLARIZER_POSITIONS], axis=-1)
Image.fromarray((rgb * 255).astype(np.uint8)).save(config.HDR_DIR / "hdr_ldic_preview.png")
plt.figure(figsize=(8, 5.5)); plt.imshow(rgb[::4, ::4]); plt.axis("off"); plt.title("LDIC HDR preview")